In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("UMUD_ROOT", "/kaggle/input"))       # Kaggle puts the competition data here
DATA = next(ROOT.rglob("sample_submission.csv")).parent
APO_IMG = next(DATA.glob("apo_imgs_v1/*"));   APO_MASK = next(DATA.glob("apo_masks_v1/*"))
FASC_IMG = next(DATA.glob("fasc_imgs_v1/*")); FASC_MASK = next(DATA.glob("fasc_masks_v1/*"))
TEST = next(DATA.glob("test_images_v2/*"))
print("data folders:", sorted(p.name for p in DATA.iterdir()))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from PIL import Image
from scipy import ndimage as ndi

StopIteration: 

In [ ]:
rows = []
for folder in [APO_IMG, APO_MASK, FASC_IMG, FASC_MASK, TEST]:
    files = [f for f in folder.iterdir() if f.suffix in (".tif", ".png")]
    rows.append({"folder": folder.parent.name, "files": len(files),
                 "types": ", ".join(sorted({f.suffix for f in files}))})
pd.DataFrame(rows)

In [ ]:
def sizes(folder, n=200):
    files = sorted(f for f in folder.iterdir() if f.suffix in (".tif", ".png"))[:n]
    return pd.Series([Image.open(f).size for f in files]).value_counts().head(4)

for folder in [APO_IMG, APO_MASK, TEST]:
    print(folder.parent.name, dict(sizes(folder)))

In [ ]:
def overlay(img_dir, mask_dir, name, ax, color):
    img = Image.open(img_dir / name).convert("L")
    mask = np.asarray(Image.open(mask_dir / name).convert("L").resize(img.size, Image.NEAREST)) > 0
    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(~mask, mask), cmap=color, alpha=0.6, vmin=0, vmax=1)
    ax.set_title(name, fontsize=9); ax.axis("off")

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, name in zip(axes[0], ["image_0000.tif", "image_0300.tif", "image_0700.tif"]):
    overlay(APO_IMG, APO_MASK, name, ax, "autumn")
for ax, name in zip(axes[1], ["image_0000.tif", "image_1500.tif", "image_2500.tif"]):
    overlay(FASC_IMG, FASC_MASK, name, ax, "cool")
axes[0, 0].set_ylabel("aponeuroses"); axes[1, 0].set_ylabel("fascicles")
plt.tight_layout(); plt.show()

In [ ]:
names = sorted(f.name for f in TEST.iterdir() if f.suffix in (".tif", ".png"))
small = [np.asarray(Image.open(TEST / n).convert("L").resize((200, 150)), float) for n in names]
diff = np.array([np.mean(np.abs(a - b)) for a, b in zip(small[:-1], small[1:])])
print(len(names), "test images;", np.sum(diff < 6), "pairs of neighbors are near copies (frames of one video)")

start = int(np.argmax(np.convolve(diff < 6, np.ones(4), "valid") == 4))   # first run of 5 near-identical frames
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, n in zip(axes, names[start:start + 5]):
    ax.imshow(Image.open(TEST / n).convert("L"), cmap="gray"); ax.set_title(n, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
sample = pd.read_csv(DATA / "sample_submission.csv", sep=None, engine="python", encoding="utf-8-sig")
print(sample.shape)          # only 2 example rows: the format, not a file to submit
sample

In [ ]:
sub = pd.DataFrame({"image_id": names,                 # all 309 test files, .tif and .png
                    "pa_deg": 15.0, "fl_mm": 75.0, "mt_mm": 19.0})   # placeholders near the example values
sub.to_csv("submission.csv", index=False)               # commas, which the grader reads
print(sub.shape); print(open("submission.csv").read()[:110])

In [ ]:
def load_pair(img_dir, mask_dir, name):
    """Grayscale image and its mask as a True/False array. Masks come in other sizes, so resize to the image."""
    img = Image.open(img_dir / name).convert("L")
    mask = Image.open(mask_dir / name).convert("L").resize(img.size, Image.NEAREST)
    return np.asarray(img, float), np.asarray(mask) > 0


def fit_line(rows, cols):
    """Least-squares line row = b0 + b1 * col. Returns b0, b1 and the angle in degrees (up is positive)."""
    b1, b0 = np.polyfit(cols, rows, 1)
    return b0, b1, -np.degrees(np.arctan(b1))


def aponeuroses(mask):
    """The two largest bands in an aponeurosis mask, top (superficial) first."""
    lab, n = ndi.label(mask)
    if n < 2:
        return None
    sizes = ndi.sum(mask, lab, range(1, n + 1))
    two = np.argsort(sizes)[-2:] + 1
    two = sorted(two, key=lambda k: ndi.center_of_mass(lab == k)[0])
    return [lab == k for k in two]


def thickness(sup, deep):
    """Muscle thickness in pixels: for each column, the gap between the bottom of the top band and the top of the deep band."""
    both = sup.any(axis=0) & deep.any(axis=0)                      # columns where both bands are drawn
    sup_bottom = sup.shape[0] - 1 - np.argmax(sup[::-1], axis=0)
    deep_top = np.argmax(deep, axis=0)
    gaps = (deep_top - sup_bottom)[both]
    return np.median(gaps) if both.sum() > 0.3 * sup.shape[1] else np.nan


def fascicle_pieces(mask, min_pixels=30):
    """Pixels of each drawn fascicle piece. The lines are 1 pixel thin and break apart when resized,
    so thicken them by 2 pixels first (binary dilation), then label the connected pieces."""
    lab, n = ndi.label(ndi.binary_dilation(mask, iterations=2))
    pieces = [np.nonzero(lab == k) for k in range(1, n + 1)]
    return [(r, c) for r, c in pieces if len(r) >= min_pixels and np.ptp(c) > 5]


def fascicle_angles(mask):
    """One least-squares line per fascicle piece; returns their angles in degrees."""
    return [fit_line(r, c)[2] for r, c in fascicle_pieces(mask)]

In [ ]:
img, mask = load_pair(APO_IMG, APO_MASK, "image_0000.tif")
sup, deep = aponeuroses(mask)
print("image", img.shape, "| mask pixels", mask.sum(), "| superficial", sup.sum(), "| deep", deep.sum())

fig, ax = plt.subplots(figsize=(8, 5.3))
ax.imshow(img, cmap="gray")
ax.imshow(np.ma.masked_where(~sup, sup), cmap="autumn", alpha=0.7)
ax.imshow(np.ma.masked_where(~deep, deep), cmap="winter", alpha=0.7)
ax.axis("off"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.3))
ax.imshow(img, cmap="gray")
for band, color in [(sup, "orange"), (deep, "deepskyblue")]:
    rows, cols = np.nonzero(band)
    b0, b1, angle = fit_line(rows, cols)
    xs = np.array([cols.min(), cols.max()])
    ax.plot(xs, b0 + b1 * xs, color=color, lw=2.5)
    print(f"{'deep' if band is deep else 'superficial':12s} row = {b0:6.1f} + {b1:+.4f} * col   angle {angle:+.2f} deg")
mt_px = thickness(sup, deep)
print(f"muscle thickness = {mt_px:.0f} px = {mt_px / img.shape[0]:.3f} of the image height")
ax.axis("off"); plt.show()

In [ ]:
img, mask = load_pair(FASC_IMG, FASC_MASK, "image_1500.tif")
print("pieces before thickening:", ndi.label(mask)[1])
pieces = fascicle_pieces(mask)
angles = fascicle_angles(mask)
print(f"pieces after thickening: {len(pieces)}; median angle {np.median(angles):.1f} deg")

fig, ax = plt.subplots(figsize=(8, 5.3))
ax.imshow(img, cmap="gray")
for rows, cols in pieces:
    b0, b1, _ = fit_line(rows, cols)
    xs = np.array([cols.min(), cols.max()])
    ax.plot(xs, b0 + b1 * xs, color="yellow", lw=2)
ax.axis("off"); plt.show()

In [ ]:
rng = np.random.default_rng(6401)
n = 400
apo_rows = []
for f in rng.choice(sorted(APO_IMG.glob("*.tif")), n, replace=False):
    img, mask = load_pair(APO_IMG, APO_MASK, f.name)
    bands = aponeuroses(mask)
    if bands is None:
        continue
    rows, cols = np.nonzero(bands[1])
    apo_rows.append({"name": f.name, "height": img.shape[0], "mt_frac": thickness(*bands) / img.shape[0],
                     "deep_angle": fit_line(rows, cols)[2]})
apo = pd.DataFrame(apo_rows).dropna()

fasc_rows = []
for f in rng.choice(sorted(FASC_IMG.glob("*.tif")), n, replace=False):
    img, mask = load_pair(FASC_IMG, FASC_MASK, f.name)
    a = fascicle_angles(mask)
    if a:
        fasc_rows.append({"name": f.name, "fasc_angle": np.median(a), "pieces": len(a)})
fasc = pd.DataFrame(fasc_rows)

apo.to_csv("apo_targets.csv", index=False); fasc.to_csv("fasc_targets.csv", index=False)
print(len(apo), "aponeurosis images,", len(fasc), "fascicle images with numbers")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].hist(apo["mt_frac"], bins=40); axes[0].set_xlabel("muscle thickness / image height")
axes[1].hist(apo["deep_angle"], bins=40); axes[1].set_xlabel("deep aponeurosis angle (deg)")
axes[2].hist(fasc["fasc_angle"], bins=40); axes[2].set_xlabel("median fascicle angle (deg)")
plt.tight_layout(); plt.show()
pd.concat([apo[["mt_frac", "deep_angle"]].describe(), fasc[["fasc_angle"]].describe()], axis=1).round(3)

Feature and Regression

In [ ]:
def load_pair(img_dir, mask_dir, name):
    """Grayscale image and its mask as a True/False array. Masks come in other sizes, so resize to the image."""
    img = Image.open(img_dir / name).convert("L")
    mask = Image.open(mask_dir / name).convert("L").resize(img.size, Image.NEAREST)
    return np.asarray(img, float), np.asarray(mask) > 0


def fit_line(rows, cols):
    """Least-squares line row = b0 + b1 * col. Returns b0, b1 and the angle in degrees (up is positive)."""
    b1, b0 = np.polyfit(cols, rows, 1)
    return b0, b1, -np.degrees(np.arctan(b1))


def aponeuroses(mask):
    """The two largest bands in an aponeurosis mask, top (superficial) first."""
    lab, n = ndi.label(mask)
    if n < 2:
        return None
    sizes = ndi.sum(mask, lab, range(1, n + 1))
    two = np.argsort(sizes)[-2:] + 1
    two = sorted(two, key=lambda k: ndi.center_of_mass(lab == k)[0])
    return [lab == k for k in two]


def thickness(sup, deep):
    """Muscle thickness in pixels: for each column, the gap between the bottom of the top band and the top of the deep band."""
    both = sup.any(axis=0) & deep.any(axis=0)                      # columns where both bands are drawn
    sup_bottom = sup.shape[0] - 1 - np.argmax(sup[::-1], axis=0)
    deep_top = np.argmax(deep, axis=0)
    gaps = (deep_top - sup_bottom)[both]
    return np.median(gaps) if both.sum() > 0.3 * sup.shape[1] else np.nan


def fascicle_pieces(mask, min_pixels=30):
    """Pixels of each drawn fascicle piece. The lines are 1 pixel thin and break apart when resized,
    so thicken them by 2 pixels first (binary dilation), then label the connected pieces."""
    lab, n = ndi.label(ndi.binary_dilation(mask, iterations=2))
    pieces = [np.nonzero(lab == k) for k in range(1, n + 1)]
    return [(r, c) for r, c in pieces if len(r) >= min_pixels and np.ptp(c) > 5]


def fascicle_angles(mask):
    """One least-squares line per fascicle piece; returns their angles in degrees."""
    return [fit_line(r, c)[2] for r, c in fascicle_pieces(mask)]

In [ ]:
from scipy.signal import find_peaks

def row_profile(img):
    w = img.shape[1]
    return ndi.gaussian_filter1d(img[:, int(w * 0.3):int(w * 0.7)].mean(axis=1), img.shape[0] / 150)

def band_gap(img):
    prof = row_profile(img)
    peaks, info = find_peaks(prof, prominence=3, distance=img.shape[0] * 0.05)
    if len(peaks) < 2:
        return np.nan
    two = peaks[np.argsort(info["prominences"])[-2:]]
    return abs(two[1] - two[0]) / img.shape[0]

img, mask = load_pair(APO_IMG, APO_MASK, "image_0000.tif")
prof = row_profile(img)
peaks, info = find_peaks(prof, prominence=3, distance=img.shape[0] * 0.05)
two = peaks[np.argsort(info["prominences"])[-2:]]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.5, 1]})
axes[0].imshow(img, cmap="gray"); [axes[0].axhline(p, color="orange", lw=1.5) for p in two]; axes[0].axis("off")
axes[1].plot(prof, np.arange(len(prof))); axes[1].invert_yaxis(); axes[1].plot(prof[two], two, "o", color="orange")
axes[1].set_xlabel("mean brightness of the row"); axes[1].set_ylabel("row")
plt.tight_layout(); plt.show()
print(f"feature band_gap = {band_gap(img):.3f}   target from the mask = {thickness(*aponeuroses(mask)) / img.shape[0]:.3f}")

In [ ]:
rng = np.random.default_rng(6401)
rows = []
for f in rng.choice(sorted(APO_IMG.glob("*.tif")), 600, replace=False):
    img, mask = load_pair(APO_IMG, APO_MASK, f.name)
    bands = aponeuroses(mask)
    if bands is not None:
        rows.append({"name": f.name, "band_gap": band_gap(img), "mt_frac": thickness(*bands) / img.shape[0]})
mt = pd.DataFrame(rows).dropna()
train = mt.sample(frac=0.75, random_state=1); valid = mt.drop(train.index)
print(len(train), "training,", len(valid), "validation images")
mt.head()

In [ ]:
model = sm.OLS(train["mt_frac"], sm.add_constant(train["band_gap"])).fit()
print(model.summary(slim=True))

In [ ]:
pred = model.predict(sm.add_constant(valid["band_gap"]))
mae = np.mean(np.abs(pred - valid["mt_frac"]))
mae0 = np.mean(np.abs(train["mt_frac"].mean() - valid["mt_frac"]))
print(f"validation MAE: model {mae:.4f}   predict-the-mean {mae0:.4f}")

fig, ax = plt.subplots(figsize=(5, 4.4))
ax.scatter(valid["band_gap"], valid["mt_frac"], s=14)
xs = np.linspace(0, 1, 2); ax.plot(xs, model.params["const"] + model.params["band_gap"] * xs, color="C1")
ax.set_xlim(0, 0.8); ax.set_ylim(0, 0.7)
ax.set_xlabel("band_gap (from the image)"); ax.set_ylabel("mt_frac (from the mask)"); ax.set_title("validation images")
plt.show()

In [ ]:
def orientation_hist(img, bins=18):
    g = np.asarray(Image.fromarray(img.astype(np.uint8)).resize((img.shape[1] // 4, img.shape[0] // 4)), float)
    h, w = g.shape
    g = ndi.gaussian_filter(g[int(h * .15):int(h * .85), int(w * .2):int(w * .8)], 1.5)
    gx, gy = ndi.sobel(g, 1), ndi.sobel(g, 0)
    jxx, jyy, jxy = (ndi.gaussian_filter(a, 4) for a in (gx * gx, gy * gy, gx * gy))
    angle = -(((0.5 * np.degrees(np.arctan2(2 * jxy, jxx - jyy)) + 90) + 90) % 180 - 90)  # stripe direction, up = +
    strength = np.sqrt((jxx - jyy) ** 2 + 4 * jxy ** 2) / (jxx + jyy + 1e-9)
    hist, _ = np.histogram(angle, bins=bins, range=(-90, 90), weights=strength)
    return hist / hist.sum()

rows = []
for f in rng.choice(sorted(FASC_IMG.glob("*.tif")), 600, replace=False):
    img, mask = load_pair(FASC_IMG, FASC_MASK, f.name)
    a = fascicle_angles(mask)
    if a:
        rows.append({"name": f.name, "fasc_angle": np.median(a), **{f"h{k}": v for k, v in enumerate(orientation_hist(img))}})
fa = pd.DataFrame(rows)
hcols = [c for c in fa.columns if c.startswith("h")]
print(len(fa), "fascicle images,", len(hcols), "features")

In [ ]:
tr = fa.sample(frac=0.75, random_state=1); va = fa.drop(tr.index)
m_ang = sm.OLS(tr["fasc_angle"], sm.add_constant(tr[hcols[:-1]])).fit()     # drop one bin: the 18 sum to 1
p_ang = m_ang.predict(sm.add_constant(va[hcols[:-1]]))
print(f"multiple regression, {len(hcols) - 1} inputs: R-squared (training) {m_ang.rsquared:.2f}")
print(f"validation MAE: model {np.mean(np.abs(p_ang - va['fasc_angle'])):.2f} deg   "
      f"predict-the-mean {np.mean(np.abs(tr['fasc_angle'].mean() - va['fasc_angle'])):.2f} deg")

In [ ]:
names = sorted(f.name for f in TEST.iterdir() if f.suffix in (".tif", ".png"))
pa_pred = []
for name in names:
    img = np.asarray(Image.open(TEST / name).convert("L"), float)
    h = orientation_hist(img)
    pa_pred.append(m_ang.predict(np.r_[1, h[:-1]].reshape(1, -1))[0])
sub = pd.DataFrame({"image_id": names,
                    "pa_deg": np.clip(np.abs(pa_pred), 0, 45),   # pennation angle: fascicle vs a near-horizontal deep band
                    "fl_mm": 75.0, "mt_mm": 19.0})               # still placeholders: millimetres need each scanner's scale
sub.to_csv("submission.csv", index=False)
print(sub.shape); print(sub.head())
print(f"predicted pa_deg on the test set: mean {sub['pa_deg'].mean():.1f}, sd {sub['pa_deg'].std():.1f}"
      f"   (training fascicle angles: mean {fa['fasc_angle'].mean():.1f})")